# Synthesis (Second Covariate Analysis) [Version updated: 13-07-2026]

## How to Use This Notebook

**1. Follow the numbered steps in order.**  
Each section builds upon the previous one, from setup, data loading, and climatology computation, to event analysis and visualization.

**2. Look for <font color="orange"> Orange cells  </font> and code cells marked as <font color="lightgreen">##### (User selection) ##### </font>:** 
| <font color="orange"> Orange cells  </font> | <font color="orange"> Need user intervention </font>|
| ----------- | ----------- |
| <font color="green">**Green cells** </font> | <font color="green">**Run automatically on user input provided in the orange cells and should not be adjusted in most cases** </font>|


**3. Run cells sequentially.**  
Start from the top and execute each cell (`Shift` + `Enter`).  

# <font color="black"> Kernel Selection</font>

**Please ensure the kernel is set to R.** If the top-right corner does not say **'R'** (or your specific R environment name):
1. Click on the kernel name in the **top-right corner**.
2. Select an appropriate **R kernel** from the dropdown menu.

In [ ]:
remotes::install_github("maris-development/rwwa@236d9a6b4a201eca1f28001b5535341022f5aeaf")
library(rwwa)

### <font color="orange"> User Input </font>

In [ ]:
your_save_directory <- "./data"

# Synthesise results from observations and climate models

### <font color="green"> Import Observations </font>

In [ ]:
observations_file_name <- "res-obs_era5.csv"

models_file_name <- "res-models-Neut.csv"

### <font color="green"> Autorun Cells </font>

In [ ]:
observations_path <- file.path(your_save_directory, observations_file_name)

# load the observational results
df_obs <- read.csv(observations_path, row.names = "X")

In [ ]:
df_obs

## Import Models

In [ ]:
models_path = file.path(your_save_directory, models_file_name)

# load the climate model results
df_models <- read.csv(models_path, row.names = "X")

In [ ]:
options(width = 10000)
options(max.print = 10000)
df_models

### Filter models

### <font color="orange"> Filter out excluded models </font>

If fewer than two models pass the model validation test, the protocol states that there are insufficient results to formulate a robust attribution statement. In such cases, synthesized values are not to be calculated, and the team must decide which results will be presented, if any. Suggested considerations of information to present are:
- Results for the observed trend;
- Statement about the validation of the models.
  
**Note:**
- The first cell manually excludes specific models by setting their Include status to FALSE.
- The second cell filters the dataset to remove all models where the Include flag is FALSE.

In [ ]:
################## (User selection) #####################
# Manually exclude specific models for testing purposes or if they have unresolvable infinite PR values. Otherwise, keep models_to_exclude empty 
models_to_exclude <- c() #(e.g., "access_cm2")

#########################################################

df_models[models_to_exclude, "Include"] <- FALSE

filter_models <- function(df, include = c("true", "false", "both")) {
    include <- match.arg(include)
    if (include == "true") {
        df <- df[df$Include == TRUE, ]
    } else if (include == "false") {
    df <- df[df$Include == FALSE, ]
    } 
    df
}

In [ ]:
# options: "true", "false", "both".
# "true" includes only models where Include == TRUE, "false" includes only models where Include == FALSE, "both" includes all models.
models_to_include <- "true"

### <font color="green"> Autorun Cells </font>

In [ ]:
df_models$Include <- as.logical(df_models$Include)
# Only models with Include == TRUE
df_models <- filter_models(df_models, models_to_include)

In [ ]:
df_models

## Calculate Synthesis

The cells below cover steps 6.3-6.7. Return to the workflow after that.

The tables generated below summarize the statistical synthesis of observations and models. Use the following guide to interpret the variables:

* **Weighted vs. Unweighted bounds:** The tables output the weighted average including uncertainty bounds (**est**, **lower**, **upper**), as well as the uncertainty bounds of the unweighted average (white bar, **l_wb** and **u_wb**).
* **Weighted Average Availability:** By definition, observations and model synthesis are only available as weighted average (**obs_synth** and **model_synth** values are NA).
* **Unweighted Mean:** The unweighted mean is given by **$uw_mean**.
* **Observational Datasets:** **sig_obs** is only relevant if there are several observational datasets.

### <font color="orange"> Choose Synthesis Type </font>

In [ ]:
# Set the intensity synthesis type based on your data:
# Use synthesis_type <- "abs" for temperature data (absolute 'shift' fit)
# Use synthesis_type <- "rel" for precipitation data (relative 'fixeddisp' fit)
################# (User selection) ##################
synthesis_type <- "abs"
#####################################################

### <font color="green"> Autorun Cells </font>

In [ ]:
cf <- "neut"

# change in intensity from past-present
synth_dI_attr <- synthesis(obs_in = df_obs[,grepl(paste0("dI_", synthesis_type, "_", cf, "_"), colnames(df_obs))], 
                           models_in = df_models[,grepl(paste0("attr_dI.", synthesis_type), colnames(df_models))], 
                           synth_type = synthesis_type)
synth_dI_attr
# if you see error/warning messages below, you probably have infinite best estimates in your observations 

In [ ]:
# change in likelihood from past-present (the code will automatically replace any infinite PR values with estimated values following the protocol in step 6.5)
synth_PR_attr <- synthesis(obs_in = infer_infinite(df_obs[,grepl((paste0("PR_", cf, "_")), colnames(df_obs))]), 
                           models_in = infer_infinite(df_models[,grepl("attr_PR", colnames(df_models))]), 
                           synth_type = "PR")
synth_PR_attr
# if you see error/warning messages below, you probably have infinite best estimates in your observations 

### Optionally remove models from probability if all values are INF

In [ ]:
# Step 6.5: If PR has an infinite lower bound value, remove the model from both PR and Intensity to ensure the final report only shows fully quantified models.
remove_inf <- TRUE

if (remove_inf) {
    
    # Identify which models have infinite values in PR (Attribution)
    # We look for infinite values in 'est' or 'lower'
    bad_models_attr <- synth_PR_attr$df$model[is.infinite(synth_PR_attr$df$est) | is.infinite(synth_PR_attr$df$lower)]
    all_bad_models_clean <- gsub("*", "", bad_models_attr, fixed = TRUE)

    # We also clean the model names in the dataframes to ensure perfect matching
    synth_PR_attr$df$model <- gsub("*", "", synth_PR_attr$df$model, fixed = TRUE)
    synth_dI_attr$df$model <- gsub("*", "", synth_dI_attr$df$model, fixed = TRUE)

    if (length(all_bad_models) > 0) {
        # Delete from PR Attribution 
        synth_PR_attr$df <- synth_PR_attr$df[!(synth_PR_attr$df$model %in% all_bad_models_clean), ]
        
        # Delete from Intensity (dI) Attribution 
        synth_dI_attr$df <- synth_dI_attr$df[!(synth_dI_attr$df$model %in% all_bad_models_clean), ]
        
        message("Deleted models with infinite PR values from all plots: ", paste(all_bad_models, collapse = ", "))
    }
}

### Combine synthesis results into table

In [ ]:
# function for extracting the correct rows from the synthesis

extract_row <- function(df, group, best = TRUE, use_wb = FALSE) {
  row <- df[df$group == group, ]

  data.frame(
    best = if (best) row$est else NA_real_,
    low  = if (use_wb) row$l_wb else row$lower,
    high = if (use_wb) row$u_wb else row$upper
  )
}


In [ ]:
#### extract rows and combine into new table

obs_PR <- extract_row(synth_PR_attr$df, "obs_synth")
obs_dI <- extract_row(synth_dI_attr$df, "obs_synth")

model_PR <- extract_row(synth_PR_attr$df, "model_synth")
model_dI <- extract_row(synth_dI_attr$df, "model_synth")

syn_w_PR <- extract_row(synth_PR_attr$df, "synth")
syn_w_dI <- extract_row(synth_dI_attr$df, "synth")

syn_uw_PR <- extract_row(synth_PR_attr$df, "synth", best = FALSE, use_wb = TRUE)
syn_uw_dI <- extract_row(synth_dI_attr$df, "synth", best = FALSE, use_wb = TRUE)


# combine results into new table
synth_comb <- rbind(
  obs = cbind(PR = obs_PR, dI = obs_dI),
  model = cbind(PR = model_PR, dI = model_dI),
  `synthesis weighted` = cbind(PR = syn_w_PR, dI = syn_w_dI),
  `synthesis unweighted` = cbind(PR = syn_uw_PR, dI = syn_uw_dI)
#  `model future` = cbind(PR = model_f_PR, dI = model_f_dI)
)

colnames(synth_comb) <- c(
  "PR best", "PR low", "PR high",
  "dI best", "dI low", "dI high"
)

# set the 'best' values for the unweighted synthesis
synth_comb["synthesis unweighted", "PR best"] <- synth_PR_attr$uw_mean
synth_comb["synthesis unweighted", "dI best"] <- synth_dI_attr$uw_mean

synth_comb

### <font color="orange"> Optionally Use Different Filenames </font>

In [ ]:
# save all the synthesised results
write.csv(synth_dI_attr$df, file.path(your_save_directory, "synth_dI_attr_neut.csv"))
write.csv(synth_PR_attr$df, file.path(your_save_directory, "synth_PR_attr_neut.csv"))
write.csv(synth_comb, file.path(your_save_directory, "synth_comb_neut.csv"))

## Synthesis figures
To show the full extent of all bars it may be necessary to change the passing to the x-axis limits

In [ ]:
plot_synthesis <- function(synth, xlim, lwd = 10, xlab = "", main = "", add_space = T, log = NA, hide_labels = F) {

  gcols = c("obs" = adjustcolor("blue", 0.5),
            "obs_synth" = "blue",
            "models" = adjustcolor("red", 0.5),
            "model_synth" = "red",
            "synth" = "magenta")

  # determine whether to plot on log axes or not (assume not unless told otherwise)
  if(is.na(log)) {
    if(!is.null(synth$synth)) {
      if (synth$synth_type == "PR") {logaxs <- "x"} else {logaxs <- ""}
    } else {
      logaxs <- ""
    }
  } else {
    if(log) {logaxs <- "x"} else {logaxs <- ""}
  }

  if (is(synth, "list")) synth <- synth$df

  if (missing(xlim)) {
    if(logaxs == "x") {
      xlim <- exp(range(pretty(log(as.numeric(unlist(synth[,c("lower", "upper", "l_wb", "u_wb")]))))))
    } else {
      xlim <- range(pretty(as.numeric(unlist(synth[,c("lower", "upper", "l_wb", "u_wb")]))))
    }
  }

  # relabel groups if needed (eg. is using results from climate explorer)
  if(is.numeric(synth$group)) synth$group <- names(gcols)[synth$group]

  nobs <- sum(synth$group == "obs")
  nmod <- sum(synth$group == "models")

  if(add_space & nobs > 0) {
    yy <- c(rev(0:nobs+nmod+4), rev(0:nmod+2), 0)
  } else {
    yy <- nrow(synth):1
  }

  if(logaxs == "x") {vline <- 1} else {vline <- 0}

  plot(0, type = "n", xlim = xlim, ylim = range(yy) + c(-0.5,0.5), log = logaxs,
       yaxt = "n", ylab = "", xlab = xlab, main = main)

  grid(ny = NA, col = adjustcolor("black", 0.1), lty = 1)
  abline(v = vline, lty = 2)

  gcols <- gcols[synth$group]

  #segments(y0 = yy, x0 = synth$l_wb, x1 = synth$u_wb, lwd = lwd, col = "black", lend = 2)
  #segments(y0 = yy, x0 = synth$l_wb, x1 = synth$u_wb, lwd = lwd-2, col = "white", lend = 2)
  #segments(y0 = yy, x0 = synth$lower, x1 = synth$upper, lwd = lwd, col = gcols, lend = 1)
  h1 <- 0.18 
  rect(xleft   = synth$l_wb,ybottom = yy - h1,xright= synth$u_wb, ytop= yy + h1,col="white",border= "black",lwd= 1)
  rect(xleft   = synth$lower,ybottom = yy - h1,xright  = synth$upper,ytop= yy + h1,col= gcols,border= NA)

  points(synth$est, yy, pch = 21, bg = gcols, lwd = 2, cex = lwd/10)

  if(!hide_labels) axis(2, at = yy, labels = synth$model, las = 1)
}

In [ ]:
prep_rc = c(1, 2)
prep_h = 5                      # height of the figure (ins)
prep_w = 5                      # width of the figure (ins)
prep_res = 200
prep_pch = 20
prep_oma = c(0, 14, 0, 0)       # increase second number until model names fit in margin
prep_mar = c(3, .5, 2, .5)      # shouldn't need to be changed

# Change until the bars fit the plot
x_lim_dI <- range(
  synth_dI_attr$df[sapply(synth_dI_attr$df, is.numeric)],
  na.rm = TRUE
) * 1.1
x_lim_PR = range(
  synth_PR_attr$df[sapply(synth_PR_attr$df, is.numeric)],
  na.rm = TRUE
) * 1.1 # add some padding to the x-axis limits

In [ ]:
# put two figures next to each other
prep_window(prep_rc,
            h = prep_h,         # height of the figure (ins)
            w = prep_w,         # width of each panel (ins)
            res = prep_res,
            pch = prep_pch,
            oma = prep_oma,     # width of each panel (ins)
            mar = prep_mar)     # shouldn't need to be changed

# set the x-axis (xlim) so that both the past & future changes use the same scaling
plot_synthesis(synth_dI_attr, lwd=12, add_space = F, main = "(a) Change in intensity neut", xlim = x_lim_dI)
plot_synthesis(synth_PR_attr, lwd=12, add_space = F, hide_labels = T, main = "(b) Probability ratio neut", xlim = x_lim_PR)

In [ ]:
attr_export_path <- file.path(your_save_directory, "synth-fig_attr_neut.png")

png(attr_export_path, height = prep_h, width  = prep_w * prep_rc[2], units  = "in", res    = prep_res); 
par(mfrow = prep_rc, oma = prep_oma, mar = prep_mar, pch = prep_pch); {
    # set the x-axis (xlim) so that both the past & future changes use the same scaling
    plot_synthesis(synth_dI_attr, add_space = F, main = "(a) Change in intensity neut", xlim = x_lim_dI)
    plot_synthesis(synth_PR_attr, add_space = F, hide_labels = T, main = "(b) Probability ratio neut", xlim = x_lim_PR)
}; dev.off()

## Optionally calculate the inverse PR
For communication purposes it may be useful to calculate the inverse PR in case average PRs are below one.

### Combined

In [ ]:
# Select columns containing "PR_"
pr_cols <- grepl("PR ", names(synth_comb))
df_synth_comb_inverse  <- synth_comb

# Invert values (1 / value)
df_synth_comb_inverse[, pr_cols] <- 1 / df_synth_comb_inverse[, pr_cols]

names(df_synth_comb_inverse) <- gsub("PR", "InvPR", names(df_synth_comb_inverse))
df_synth_comb_inverse[] <- lapply(df_synth_comb_inverse, function(x) if(is.numeric(x)) signif(x, 3) else x)

df_synth_comb_inverse

## Table for Scientific Report
- Save output table for Scientific report attribution Table_5_1.csv 
- Copy this table to the Scientific report Section 5.

In [ ]:
##################################
#Scientific report:
#Table 5.1.
#Probability ratio (PR) and change in intensity (𝛥I) for a  X-year return period of VARIABLE. Results are shown for models that passed the evaluation tests (a) from pre-industrial climate to the present (-1.3°C) and (b) from the present to 2.6°C above pre-industrial climate.
##################################

# 1.determine the unit to be used based on the synthesis type selection
# --> absolute change for temperature (shift)
# --> relative change for precipitation (fixeddisp)
unit_label <- if (synthesis_type == "abs") "(°C)" else "(%)"


# 2. extract the relevant rows from df_models that contains the values BEFORE the infinite replacement takes place
#applicable only to PR not ∆I
attr_PR <- data.frame(
  model = rownames(df_models), #extract the model names from the row names 
  est = df_models$attr_PR_est,
  lower = df_models$attr_PR_lower,
  upper = df_models$attr_PR_upper,
  stringsAsFactors = FALSE  #prevent from converting the model name strings into categorical variables
)

#∆I are not affected by the infinite replacement and are drawn from the synthesis data frames
attr_dI <- synth_dI_attr$df[synth_dI_attr$df$group == "models", c("model", "est", "lower", "upper")]

# 3.rename the column names with units 
#PR does not have units
colnames(attr_PR) <- c("Model", "Attr PR neut est (-)","Attr PR neut lower (-)", "Attr PR neut higher (-)")
# ∆I has units: either °C or % based on the synthesis choice 
#-->Attribution 
colnames(attr_dI) <- c("Model", paste("Attr ∆I neut est", unit_label),
                                paste("Attr ∆I neut lower", unit_label),
                                paste("Attr ∆I neut higher", unit_label))

# 4.merge all into one master table based on the shared model names
master_table <- merge(attr_PR, attr_dI, by="Model")

# 5. round to 2 decimal places while preserving the infinite values
#selecting the numeric columns to apply the rounding
numeric_cols <- sapply(master_table, is.numeric)
master_table[numeric_cols] <- lapply(master_table[numeric_cols], function(x) {
  ifelse(is.infinite(x), x, round(x,2)) #if the value is infinite, return it unchanged, otherwise round it to 2 decimal places 
  })

# 6. save to the data with the name "Table_5_1.csv"
write.csv(master_table, file.path(your_save_directory, "Table_5_1_neut.csv"),
          row.names=FALSE, fileEncoding = "UTF-8")

#print the table 
master_table
